In [ ]:
# ============================================
# SESSION 6
# DECISION TREE - GINI, ENTROPY & PRUNING
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.tree import (
    DecisionTreeClassifier,
    plot_tree
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================
# 1. LOAD DATASET
# ============================================

CSV_PATH = "/content/placement_predict_50k Dataset.csv"

df = pd.read_csv(
    CSV_PATH,
    on_bad_lines="skip",
    engine="python"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)


# ============================================
# 2. REMOVE EMPTY ROWS
# ============================================

df = df.dropna(how="all")

print("Shape after cleaning:", df.shape)


# ============================================
# 3. TARGET VARIABLE
# ============================================

target = "PlacementStatus"

# Remove rows where target is missing
df = df.dropna(subset=[target])

X = df.drop(columns=[target])
y = df[target]

print("\nTarget variable:", target)

print("\nClass distribution:")
print(y.value_counts())


# ============================================
# 4. IDENTIFY COLUMN TYPES
# ============================================

numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumerical columns:")
print(numeric_features)

print("\nCategorical columns:")
print(categorical_features)


# ============================================
# 5. PREPROCESSING
# ============================================

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)


# ============================================
# 6. TRAIN-VALIDATION SPLIT
# ============================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])


# ============================================
# 7. TRANSFORM DATA
# ============================================

X_train_processed = preprocessor.fit_transform(
    X_train
)

X_val_processed = preprocessor.transform(
    X_val
)

feature_names = preprocessor.get_feature_names_out()

print("\nProcessed training shape:",
      X_train_processed.shape)


# ============================================
# 8. DECISION TREE - GINI
# ============================================

dt_gini = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

dt_gini.fit(
    X_train_processed,
    y_train
)

gini_train_pred = dt_gini.predict(
    X_train_processed
)

gini_val_pred = dt_gini.predict(
    X_val_processed
)

print("\n============================================")
print("DECISION TREE - GINI")
print("============================================")

print(
    "Training Accuracy:",
    round(
        accuracy_score(
            y_train,
            gini_train_pred
        ),
        4
    )
)

print(
    "Validation Accuracy:",
    round(
        accuracy_score(
            y_val,
            gini_val_pred
        ),
        4
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        gini_val_pred
    )
)


# ============================================
# 9. DECISION TREE - ENTROPY
# ============================================

dt_entropy = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

dt_entropy.fit(
    X_train_processed,
    y_train
)

entropy_train_pred = dt_entropy.predict(
    X_train_processed
)

entropy_val_pred = dt_entropy.predict(
    X_val_processed
)

print("\n============================================")
print("DECISION TREE - ENTROPY")
print("============================================")

print(
    "Training Accuracy:",
    round(
        accuracy_score(
            y_train,
            entropy_train_pred
        ),
        4
    )
)

print(
    "Validation Accuracy:",
    round(
        accuracy_score(
            y_val,
            entropy_val_pred
        ),
        4
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        entropy_val_pred
    )
)

Dataset loaded successfully!
Shape: (50000, 32)
Shape after cleaning: (50000, 32)

Target variable: PlacementStatus

Class distribution:
PlacementStatus
1    32856
0    17144
Name: count, dtype: int64

Numerical columns:
['StudentID', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'IsAnomaly', 'Salary Package']

Categorical columns:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'CGPA_Tier']

Training samples: 40000
Validation samples: 10000

Processed training shape: (40000, 55)

DECISION TREE - GINI
Training Accuracy: 0.9988
Validation Accuracy: 0.9981

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      3429